# OpusClip — Clip Selection Debug Notebook

**هذا الـ notebook يطابق كود الـ Production تمامًا خطوة بخطوة**
مع إضافة `print()` و `time.perf_counter()` قبل وبعد كل دالة دون تغيير أي Logic.

الهدف: Debug دقيق لمسار Clip Selection لاكتشاف مكان المشكلة.

---
### Runtime Setup
1. **Runtime -> Change runtime type -> T4 GPU** (recommended)
2. Set **OPUSCLIP_API_KEY** in Colab Secrets or Cell 2

---
## Cell 1 — Environment & Dependencies

In [ ]:
import subprocess, sys, os, pathlib, json, importlib, time, warnings, re, functools
from typing import Any, Callable, TypeVar, Literal
from dataclasses import dataclass, fields
from abc import ABC, abstractmethod

python = sys.executable

def _sh(cmd, label=""):
    print(f"  -> {label or cmd[:70]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and r.stderr.strip():
        print(f"    ! {r.stderr.strip()[:200]}")
    return r

# --- ffmpeg ---
if _sh("ffmpeg -version").returncode != 0:
    _sh("apt-get install -y -q ffmpeg > /dev/null 2>&1")
ffmpeg_ver = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0]
print(f"  ffmpeg    == {ffmpeg_ver}")

# --- yt-dlp ---
if _sh("yt-dlp --version").returncode != 0:
    _sh(f'"{python}" -m pip install -q yt-dlp')
ytdlp_ver = subprocess.run(["yt-dlp", "--version"], capture_output=True, text=True).stdout.strip()
print(f"  yt_dlp    == {ytdlp_ver}")

# --- opusclip from GitHub tag v1.0.0 ---
try:
    import opusclip
except ModuleNotFoundError:
    _sh(f'"{python}" -m pip install -q git+https://github.com/ABo-EsMaiL/OpusClip.git@v1.0.0')
    importlib.invalidate_caches()
    import opusclip
print(f"  opusclip  == {opusclip.__version__}  ({opusclip.__file__})")

# --- fonts ---
_sh("apt-get install -y -q fonts-noto-core fonts-noto-extra fonts-dejavu-core > /dev/null 2>&1", "fonts")

# --- torch / CUDA ---
import torch
print(f"  Python    == {sys.version.split()[0]}")
print(f"  torch     == {torch.__version__}")
print(f"  CUDA      == {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU       == {torch.cuda.get_device_name(0)}")

# --- openai SDK ---
from openai import OpenAI, OpenAIError
print(f"  openai    == ok")

print("\nAll dependencies ready.")

---
## Cell 2 — Configuration

يطابق `PipelineConfig.from_env()` في الـ Production.

In [ ]:
from opusclip.config import PipelineConfig

# --- API Key ---
try:
    from google.colab import userdata
    os.environ.setdefault("OPUSCLIP_API_KEY", userdata.get("OPUSCLIP_API_KEY"))
    print("API key loaded from Colab Secrets")
except (ImportError, ValueError, KeyError):
    os.environ.setdefault("OPUSCLIP_API_KEY", "")

os.environ.setdefault("LLM_BASE_URL", "https://opencode.ai/zen/v1")
os.environ.setdefault("LLM_MODEL", "deepseek-v4-flash-free")

if not os.environ.get("OPUSCLIP_API_KEY"):
    raise RuntimeError("OPUSCLIP_API_KEY is not set.")

config = PipelineConfig.from_env()
print(f"PipelineConfig ({len(fields(config))} fields):")
for f in fields(config):
    val = getattr(config, f.name)
    print(f"  {f.name:30s} = {val}")

---
## Cell 3 — Video Input & Transcription

نحصل على `TranscriptResult` الذي سيدخل إلى `select_clips()`.

In [ ]:
from opusclip.subprocess_utils import run_ffmpeg
from opusclip.transcription.whisper_provider import WhisperProvider
from opusclip.transcription.base import TranscriptResult, TranscriptSegment, WordInfo
from opusclip.transcription.word_repair import fill_missing_words

def _sh(cmd, label=""):
    print(f"  -> {label or cmd[:70]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and r.stderr.strip():
        print(f"    ! {r.stderr.strip()[:200]}")
    return r

INPUT_MODE  = "youtube"
YOUTUBE_URL = "https://youtu.be/OwDU3GH9cGI?si=MeHrfxVtdIUK5u9c"

work_dir = pathlib.Path.cwd()
VIDEO_PATH = None
if INPUT_MODE == "upload":
    from google.colab import files as _cf
    _uploaded = _cf.upload()
    VIDEO_PATH = str(work_dir / list(_uploaded.keys())[0])
elif INPUT_MODE == "youtube":
    dest = work_dir / "source_video.mp4"
    VIDEO_PATH = str(dest)
    if not dest.exists():
        _sh('yt-dlp --format "bestvideo[height<=1080][ext=mp4]+bestaudio[ext=m4a]/best[height<=1080]" '
            '--merge-output-format mp4 --no-playlist '
            f'--output "{VIDEO_PATH}" "{YOUTUBE_URL}"')

print(f"\n  Video : {VIDEO_PATH}")
print(f"  Size  : {os.path.getsize(VIDEO_PATH) / 1_048_576:.1f} MB")

import cv2
cap = cv2.VideoCapture(VIDEO_PATH)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
nf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
dur = nf / fps if fps else 0
cap.release()
print(f"  Res   : {w}x{h}, fps={fps:.2f}, dur={dur:.1f}s")

output_dir = pathlib.Path(str(config.output_dir))
output_dir.mkdir(exist_ok=True)

# --- Audio extraction ---
audio_path = output_dir / "audio.wav"
t0 = time.perf_counter()
run_ffmpeg(["-y", "-i", VIDEO_PATH, "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", str(audio_path)])
t1 = time.perf_counter()
print(f"\n  Audio extract : {t1 - t0:.2f}s, size={audio_path.stat().st_size / 1_048_576:.1f} MB")

# --- Whisper transcription ---
whisper = WhisperProvider(model_size=config.whisper_model, device=config.whisper_device)
t0 = time.perf_counter()
whisper_result = whisper.transcribe(audio_path, "ar")
t1 = time.perf_counter()
print(f"  Transcription : {t1 - t0:.2f}s")
print(f"  Language      : {whisper_result.language}")
print(f"  Segments      : {len(whisper_result.segments)}")
print(f"  Words         : {len(whisper_result.words)}")
transcript_text = " ".join(s.text for s in whisper_result.segments)
print(f"  Total chars   : {len(transcript_text)}")

# --- Transcription repair ---
t0 = time.perf_counter()
repaired_words = fill_missing_words(whisper_result)
t1 = time.perf_counter()
print(f"\n  Repair time   : {t1 - t0:.4f}s")
print(f"  Repaired words: {len(repaired_words)}")

# --- Build segments list for compression ---
segments: list[TranscriptSegment] = list(whisper_result.segments)

# Save transcript JSON (same format as pipeline.py step 3)
transcript_data = {
    "segments": [(s.id, s.text, s.start, s.end) for s in segments],
    "words": [(w.word, w.start, w.end, w.probability) for w in whisper_result.words],
    "repaired_words": [(w.word, w.start, w.end, w.probability) for w in repaired_words],
    "language": whisper_result.language,
    "duration": whisper_result.duration,
}
transcript_path = output_dir / "transcript.json"
with open(transcript_path, "w", encoding="utf-8") as f:
    json.dump(transcript_data, f, ensure_ascii=False, indent=2)
print(f"\n  Transcript saved: {transcript_path}")

print("\n=== Cell 3 complete. TranscriptResult ready for Clip Selection ===")

---
## Cell 4 — Production Code: `get_clip_selection_prompt`

هذه الدالة منسوخة حرفيًا من `src/opusclip/clip_selection/prompts.py`

بعد التنفيذ: نطبع الـ System Prompt بالكامل.

In [ ]:
# ========== PRODUCTION CODE (copied verbatim from prompts.py) ==========

def get_clip_selection_prompt(
    min_clips: int,
    max_clips: int,
    min_duration: int,
    max_duration: int,
    min_virality: int,
    lang_hint: str,
) -> str:
    return f"""You are a world-class viral content strategist for Arabic and English short-form video.

TASK: Analyze the full transcript and select the best segments for viral vertical clips.

HOW MANY CLIPS:
  You decide — between {min_clips} and {max_clips} clips, based on quality alone.
  Only include clips scoring >= {min_virality}/100. Never pad with weak content.
  Clips MAY OVERLAP if the same content serves two different story angles.

DURATION: {min_duration}–{max_duration} seconds.
  Prefer 40–90 s. Including one extra sentence is always better than cutting a thought short.

SENTENCE BOUNDARY RULE — MOST IMPORTANT:
  start = timestamp where a COMPLETE sentence/thought BEGINS (never mid-sentence)
  end   = timestamp where the COMPLETE thought is FULLY RESOLVED, including punchline
  Only use timestamps that appear in the [start-end] markers in the transcript.

VIRALITY SCORE (sum of 5 x 0–20 = 0–100):
  hook    — Do the first 5 words demand attention? Would someone stop scrolling?
  trigger — Activates: shock / awe / outrage / fear / curiosity / FOMO / relatability?
  value   — Rare knowledge, secret revealed, counterintuitive truth?
  arc     — Clear setup → tension/twist → resolution within this single clip?
  share   — Would viewers immediately tag someone or repost after watching?

TITLE LANGUAGE: Match the video language ({lang_hint}).
  Arabic content → Arabic titles. English content → English titles.

RETURN: ONLY a valid JSON array — no text before or after, no markdown fences.
[
  {{
    "clip_number"     : 1,
    "title"           : "<=80 chars · 1–2 emojis · match video language",
    "start"           : 0.0,
    "end"             : 0.0,
    "virality_score"  : 0,
    "score_breakdown" : {{"hook":0,"trigger":0,"value":0,"arc":0,"share":0}},
    "hook_line"       : "Exact first sentence that hooks the viewer",
    "reason"          : "2–3 sentences: hook, trigger, why viewers will share"
  }}
]
"""

# ========== End of production copy ==========

lang_hint = whisper_result.language or "ar"

t0 = time.perf_counter()
clip_system = get_clip_selection_prompt(
    min_clips=config.min_clips,
    max_clips=config.max_clips,
    min_duration=config.min_duration,
    max_duration=config.max_duration,
    min_virality=config.min_virality,
    lang_hint=lang_hint,
)
t1 = time.perf_counter()

print(f"  Prompt build time : {t1 - t0:.6f}s")
print(f"  System prompt len : {len(clip_system)} chars")
print()
print("=" * 60)
print("SYSTEM PROMPT (full)")
print("=" * 60)
print(clip_system)
print("=" * 60)

---
## Cell 5 — Full Transcript (No Compression)

We send the full transcript to the LLM (no compression).
Modern LLMs handle large contexts natively.

بعد التنفيذ: نطبع الـ full transcript بالكامل مع الإحصائيات.

In [ ]:
# ========== Production-style transcript build (no compression) ==========

tx_text = "\n".join(
    f"[{s.start:.1f}s-{s.end:.1f}s]: {s.text}" for s in segments
)
tx_chars = len(tx_text)

print(f"  Full transcript lines : {len(segments)}")
print(f"  Full transcript chars : {tx_chars}")
print(f"  Est. prompt tokens    : ~{(len(clip_system) + tx_chars) // 4}")
print()
print("=" * 60)
print("FULL TRANSCRIPT")
print("=" * 60)
print(tx_text)
print("=" * 60)

# Save to disk
prompt_path = output_dir / "llm_prompt.txt"
with open(prompt_path, "w", encoding="utf-8") as f:
    f.write("=== SYSTEM PROMPT ===\n")
    f.write(clip_system)
    f.write("\n\n=== USER PROMPT ===\n")
    f.write(tx_text)
print(f"\n  Prompt saved: {prompt_path}")

---
## Cell 6 — Production Code: `with_retry` + `_fetch_clips` (LLM API Call)

**أهم خلية في الـ Notebook**

هذا الكود منسوخ حرفيًا من:
- `retry_utils.py` ← `with_retry`
- `llm_selector.py` ← `_fetch_clips` (inside `select_clips`)
- `llm_selector.py` ← `_LLM_TEMPERATURE`, `self.client`, `self.model`

سنقوم بعمل API call مطابق تمامًا للـ Production،
مع طباعة كل Parameter وكل جزء من الـ Response.

In [ ]:
from opusclip.security import load_api_key
from opusclip.utils.json_utils import extract_json_array

# Production constants (moved from removed Cell 5)
_MAX_SNAP_WINDOW_S: float = 4.0
_LLM_TEMPERATURE: float = 0.20

# ========== PRODUCTION CODE (copied verbatim from retry_utils.py) ==========

T = TypeVar("T")


def with_retry(
    attempts: int = 3,
    delay_s: float = 2.0,
    backoff_factor: float = 2.0,
    exceptions: tuple[type[Exception], ...] = (Exception,),
) -> Callable[[Callable[..., T]], Callable[..., T]]:
    def decorator(func: Callable[..., T]) -> Callable[..., T]:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> T:
            current_delay = delay_s
            last_exc: Exception = RuntimeError("No attempts made")
            for attempt in range(1, attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    last_exc = exc
                    if attempt < attempts:
                        time.sleep(current_delay)
                        current_delay *= backoff_factor
            raise last_exc
        return wrapper
    return decorator


# ========== End of production copy ==========

# Build the OpenAI client (same as LLMClipSelector.__init__)
api_key = load_api_key()
client = OpenAI(api_key=api_key, base_url=config.llm_base_url)
model = config.llm_model

print("=" * 60)
print("LLM API CALL — PARAMETERS")
print("=" * 60)
print(f"  model                      : {model}")
print(f"  base_url                   : {config.llm_base_url}")
print(f"  temperature                : {_LLM_TEMPERATURE}")
print(f"  max_tokens                 : not set (default)")
print(f"  max_completion_tokens      : not set (default)")
print(f"  response_format            : not set (default)")
print(f"  extra_body                 : not set")
print(f"  timeout                    : not set (default)")
print(f"  headers                    : not set (default)")
print()
print(f"  retry.attempts             : {config.api_retry_attempts}")
print(f"  retry.delay_s              : {config.api_retry_delay_s}")
print(f"  retry.backoff_factor       : {config.api_retry_backoff_factor}")
print(f"  retry.exceptions           : OpenAIError, JSONDecodeError, ValueError")
print()
print(f"  system prompt length       : {len(clip_system)} chars")
print(f"  user prompt length         : {len(tx_text)} chars")
print(f"  total prompt length        : {len(clip_system) + len(tx_text)} chars")
print(f"  est. prompt tokens         : ~{(len(clip_system) + len(tx_text)) // 4}")
print()

user_content = f"Transcript:\n\n{tx_text}"
messages = [
    {"role": "system", "content": clip_system},
    {"role": "user", "content": user_content},
]
print(f"  messages[0] (system)       : {len(messages[0]['content'])} chars")
print(f"  messages[1] (user)         : {len(messages[1]['content'])} chars")

# ========== PRODUCTION CODE: _fetch_clips (copied verbatim from llm_selector.py) ==========

@with_retry(
    attempts=config.api_retry_attempts,
    delay_s=config.api_retry_delay_s,
    backoff_factor=config.api_retry_backoff_factor,
    exceptions=(OpenAIError, json.JSONDecodeError, ValueError),
)
def _fetch_clips() -> list[dict[str, object]]:
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=_LLM_TEMPERATURE,
    )
    raw_json = resp.choices[0].message.content or ""
    parsed = extract_json_array(raw_json)
    if parsed is None:
        raise ValueError("Failed to parse JSON array from LLM response.")
    return parsed  # type: ignore[return-value]


# ========== End of production copy ==========

# --- Execute with full instrumentation ---
print("\n" + "=" * 60)
print("LLM API CALL — EXECUTION")
print("=" * 60)
print(f"  Start time : {time.strftime('%H:%M:%S')}")
t_req0 = time.perf_counter()

try:
    clips_raw = _fetch_clips()
    t_req1 = time.perf_counter()
    print(f"  Finish time: {time.strftime('%H:%M:%S')}")
    print(f"  Latency    : {t_req1 - t_req0:.2f}s")
    print()

    # We need the raw response for full debugging.
    # _fetch_clips returns parsed JSON, so we re-fetch to capture raw response.
    print("  (Re-fetching to capture raw response details...)")
    resp_debug = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=_LLM_TEMPERATURE,
    )
    choice = resp_debug.choices[0]
    raw_content = choice.message.content or ""

    print(f"\n  finish_reason       : {choice.finish_reason}")
    if hasattr(resp_debug, "usage") and resp_debug.usage:
        print(f"  usage.prompt_tokens     : {resp_debug.usage.prompt_tokens}")
        print(f"  usage.completion_tokens : {resp_debug.usage.completion_tokens}")
        print(f"  usage.total_tokens       : {resp_debug.usage.total_tokens}")
    else:
        print(f"  usage                    : not available")
    print()
    print("=" * 60)
    print("RAW LLM RESPONSE")
    print("=" * 60)
    print(raw_content)
    print("=" * 60)

    # Save raw response
    raw_path = output_dir / "llm_raw_response.txt"
    with open(raw_path, "w", encoding="utf-8") as f:
        f.write(raw_content)
    print(f"\n  Raw response saved: {raw_path}")

except Exception as exc:
    t_req1 = time.perf_counter()
    print(f"  FAILED after {t_req1 - t_req0:.2f}s: {type(exc).__name__}: {exc}")
    import traceback
    traceback.print_exc()
    raise

---
## Cell 7 — Production Code: `extract_json_array`

منسوخة حرفيًا من `json_utils.py`.

نستخدم `clips_raw` الناتجة من `_fetch_clips()` كمدخل.

In [ ]:
# ========== PRODUCTION CODE: extract_json_array (copied verbatim from json_utils.py) ==========

def extract_json_array(text: str) -> list[dict[str, Any]] | None:
    if not text or not text.strip():
        return None
    cleaned = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned).strip()
    m = re.search(r"\[.*\]", cleaned, re.DOTALL)
    if m:
        try:
            result = json.loads(m.group())
            if isinstance(result, list):
                return result  # type: ignore[return-value]
        except (json.JSONDecodeError, ValueError):
            pass
    try:
        result = json.loads(cleaned)
        if isinstance(result, list):
            return result  # type: ignore[return-value]
    except (json.JSONDecodeError, ValueError):
        pass
    return None


# ========== End of production copy ==========

print(f"  clips_raw type   : {type(clips_raw).__name__}")
print(f"  clips_raw length : {len(clips_raw)}")
print()

for i, obj in enumerate(clips_raw):
    print(f"  Object #{i + 1}:")
    print(f"    {json.dumps(obj, ensure_ascii=False, indent=4)}")
    print()

---
## Cell 8 — Production Code: Validation + Filtering + `_snap_boundary`

منسوخ حرفيًا من `LLMClipSelector.select_clips()` (Validation + Filtering loop).

كل قيمة يتم استبعادها تُطبع مع سبب الاستبعاد.

In [ ]:
from opusclip.clip_selection.base import ClipCandidate
from opusclip.exceptions import OpusClipError, ClipSelectionError

# ========== PRODUCTION CODE: _snap_boundary (copied verbatim from llm_selector.py) ==========

def _snap_boundary(
    ts: float,
    segments: list[TranscriptSegment],
    direction: Literal["start", "end"] = "start",
    window: float = _MAX_SNAP_WINDOW_S,
) -> float:
    best, best_d = float(ts), float("inf")
    for seg in segments:
        cand = seg.start if direction == "start" else seg.end
        d = abs(cand - ts)
        if d < best_d and d <= window:
            best, best_d = cand, d
    return round(best, 3)


# ========== End of production copy ==========

# ========== PRODUCTION CODE: required fields set (copied verbatim from llm_selector.py) ==========

required = {
    "clip_number",
    "title",
    "start",
    "end",
    "virality_score",
    "score_breakdown",
    "hook_line",
    "reason",
}


# ========== End of production copy ==========

# ========== PRODUCTION CODE: Validation + filtering loop (copied verbatim from llm_selector.py) ==========

print("=" * 60)
print("VALIDATION & FILTERING")
print("=" * 60)

selected_clips: list[ClipCandidate] = []
discard_reasons: list[tuple[int, str, str]] = []  # (index, reason, title)

for idx, c in enumerate(clips_raw):
    print(f"\n  [{idx}] Processing object #{idx + 1}: {json.dumps(c, ensure_ascii=False)}")

    # --- Required fields check ---
    if not required.issubset(c.keys()):
        missing = required - c.keys()
        print(f"    >> DISCARDED: missing fields {missing}")
        discard_reasons.append((idx, "missing fields", str(missing)))
        continue

    # --- Type coercion ---
    try:
        v_score = float(c["virality_score"])  # type: ignore[arg-type]
        c_start_raw = float(c["start"])  # type: ignore[arg-type]
        c_end_raw = float(c["end"])  # type: ignore[arg-type]
    except (TypeError, ValueError) as e:
        print(f"    >> DISCARDED: type coercion failed: {e}")
        discard_reasons.append((idx, "type coercion", str(e)))
        continue

    print(f"       virality_score = {v_score}, start = {c_start_raw}, end = {c_end_raw}")

    # --- Virality filter ---
    if v_score < config.min_virality:
        print(f"    >> DISCARDED: virality {v_score} < {config.min_virality}")
        discard_reasons.append((idx, "virality", f"{v_score} < {config.min_virality}"))
        continue

    # --- Snap boundaries ---
    c_start = _snap_boundary(c_start_raw, segments, "start")
    c_end = _snap_boundary(c_end_raw, segments, "end")
    duration = round(c_end - c_start, 1)
    print(f"       snapped: {c_start_raw} -> {c_start}, {c_end_raw} -> {c_end}, duration={duration}s")

    # --- Duration filter ---
    if not (config.min_duration <= duration <= config.max_duration):
        print(f"    >> DISCARDED: duration {duration}s not in [{config.min_duration}, {config.max_duration}]")
        discard_reasons.append((idx, "duration", f"{duration}s out of range"))
        continue

    # --- Accept ---
    selected_clips.append(
        ClipCandidate(
            start=c_start,
            end=c_end,
            score=v_score,
            title=str(c["title"]),
            summary=str(c["reason"]),
        )
    )
    print(f"    >> ACCEPTED (score={v_score})")


# ========== End of production copy ==========

print()
print("=" * 60)
print("FILTERING SUMMARY")
print("=" * 60)
print(f"  Total objects from LLM    : {len(clips_raw)}")
print(f"  Validation failures       : {sum(1 for _, r, _ in discard_reasons if r == 'missing fields')}")
print(f"  Type coercion failures    : {sum(1 for _, r, _ in discard_reasons if r == 'type coercion')}")
print(f"  Discarded (virality)      : {sum(1 for _, r, _ in discard_reasons if r == 'virality')}")
print(f"  Discarded (duration)      : {sum(1 for _, r, _ in discard_reasons if r == 'duration')}")
print(f"  Total discarded           : {len(discard_reasons)}")
print(f"  Accepted (ClipCandidates) : {len(selected_clips)}")

if selected_clips:
    print()
    print("=" * 60)
    print("ACCEPTED CLIP CANDIDATES")
    print("=" * 60)
    selected_clips.sort(key=lambda x: x.score, reverse=True)
    for i, cc in enumerate(selected_clips):
        print(f"  #{i + 1}: score={cc.score}, {cc.start:.1f}-{cc.end:.1f}s, title={cc.title[:60]}")
else:
    raise ClipSelectionError("No valid clips remain after applying virality and duration constraints.")

---
## Cell 9 — Final TranscriptResult Rebuild & Summary

نعيد بناء `TranscriptResult` بنفس الطريقة التي تعمل بها `Pipeline.step_5_select_clips()`
ثم نستدعي `select_clips()` مرة أخرى عبر `LLMClipSelector` للتأكيد.

هذا يثبت أن الـ Debug Notebook ينتج نفس النتيجة تمامًا مثل الـ Production.

In [ ]:
from opusclip.clip_selection.llm_selector import LLMClipSelector

# Build TranscriptResult (same as pipeline.py step 5)
pipeline_segments = [
    TranscriptSegment(id=sid, text=stxt, start=sstart, end=send, words=[])
    for sid, stxt, sstart, send in transcript_data["segments"]
]
pipeline_words = [
    WordInfo(word=wword, start=wstart, end=wend, probability=wprob)
    for wword, wstart, wend, wprob in transcript_data["words"]
]

transcript_for_production = TranscriptResult(
    segments=pipeline_segments,
    words=pipeline_words,
    language=transcript_data["language"],
    duration=transcript_data["duration"],
)

# Call LLMClipSelector.select_clips() directly (production path)
selector = LLMClipSelector(
    api_key=load_api_key(),
    base_url=config.llm_base_url,
    model=config.llm_model,
)

print("Calling LLMClipSelector.select_clips() (production path)...")
print("This will make another API call.")
print()

t0 = time.perf_counter()
try:
    production_result = selector.select_clips(transcript_for_production, config)
    t1 = time.perf_counter()
    print(f"  select_clips() returned in {t1 - t0:.2f}s")
    print(f"  Production clip count : {len(production_result)}")
    print()
    for cc in production_result:
        print(f"  #{cc.clip_number}: score={cc.score}, {cc.start:.1f}-{cc.end:.1f}s, {cc.title[:60]}")

    # Compare with our notebook result
    print()
    print("=" * 60)
    print("COMPARISON: Notebook vs Production")
    print("=" * 60)
    print(f"  Notebook candidates    : {len(selected_clips)}")
    print(f"  Production candidates  : {len(production_result)}")
    if len(selected_clips) == len(production_result):
        match = all(
            abs(nc.start - pc.start) < 0.1 and abs(nc.end - pc.end) < 0.1
            for nc, pc in zip(selected_clips, production_result)
        )
        print(f"  Boundary match         : {'YES' if match else 'NO (values differ)'}")
    else:
        print(f"  Boundary match         : N/A (different counts)")

except ClipSelectionError as e:
    t1 = time.perf_counter()
    print(f"  FAILED after {t1 - t0:.2f}s: {e}")
    print("  (This may be due to the retry re-fetch in Cell 6 consuming the API quota.)")

---
## Summary of Saved Artifacts

| File | Cell | Description |
|------|------|-------------|
| `opusclip_output/transcript.json` | 3 | Full transcription with words and segments |
| `opusclip_output/llm_prompt.txt` | 5 | System prompt + full transcript sent to LLM |
| `opusclip_output/llm_raw_response.txt` | 6 | Raw LLM response before any parsing |

---
*هذا الـ Notebook يطابق Production Code 100%. أي اختلاف في النتائج يعني وجود مشكلة في البيئة.*